# 9.4 - Open-Source Fine-Tuning

**Module 9 - Fine-Tuning** | Netsetos GenAI Engineering

This notebook builds the open-weight fine-tuning pipeline in runnable miniature: instead of renting an A100, you run small Python harnesses that compute LoRA parameter counts, QLoRA VRAM budgets, and the decision rules for LoRA variants, alignment methods, serving frameworks, and the India open-weight stack. Every cell you actually execute here is plain Python that makes the tradeoffs concrete and checkable - no GPU required.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the open-source fine-tuning stack. On a fresh Colab or clean environment, uncomment the line before running; in a preconfigured environment it is a no-op. These are the training-side libraries a real open-weight fine-tune would use.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install transformers torch datasets "trl>=1.0" peft transformers datasets peft unsloth python-dotenv -q  # noqa


**Explanation**

A one-line dependency install, commented out by default so it does not slow re-runs in an already-provisioned environment.

**How the code works - step by step**
- **`transformers` / `torch` / `datasets`** - the base model loaders, tensor backend, and dataset plumbing.
- **`trl>=1.0`** - the trainer layer (SFTTrainer, DPO, GRPO); the `>=1.0` pin is deliberate because v1.0 renamed key arguments.
- **`peft`** - LoRA / DoRA adapters attached to the frozen base.
- **`unsloth`** - the fast-path loaders and GGUF export.
- **`python-dotenv`** - reads keys from a `.env` file.

**In one line:** installs the HuggingFace + TRL + PEFT + Unsloth training stack.

**What you'll see:** (no output - environment prep)

## Setup - provider keys

Seed empty environment variables for the three API providers so later cells can read them without a KeyError. The fine-tuning demos in this lesson are local/GPU-side and do not need a key, but the notebook keeps the standard key-loading block so it matches the rest of the course.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Environment prep, not a model call: `setdefault` writes an empty string only if the variable is not already set, so a key you exported earlier is preserved and a missing one becomes a harmless empty placeholder.

**How the code works - step by step**
- **`import os`** - access to the process environment.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** and the two siblings - register each provider key, defaulting to empty rather than overwriting an existing value.
- Keys are never hardcoded; any one provider is enough, and the comments point to where each key is issued.

**In one line:** guarantees the three key variables exist without clobbering anything already set.

**What you'll see:** (no output - environment prep)

## 1 - Count LoRA's trainable parameters

LoRA replaces a full weight update with two small matrices, so a layer that would train millions of parameters trains only a few thousand. This cell puts a number on that: how many parameters a single linear layer costs at each rank, and how a whole-model adapter still stays under half a percent of a 7B. That fraction is exactly why saved adapters are tens of megabytes instead of gigabytes.

In [ ]:
# LoRA TRAINABLE PARAMETERS - a d x k linear gets two small matrices A (r x k), B (d x r).
# So it trains r*(d + k) params instead of d*k.
def lora_params(d, k, r):
    return r * (d + k)          # A: r*k  +  B: d*r
def dense_params(d, k):
    return d * k

d = k = 4096                     # a Gemma/Llama hidden dimension
for r in (8, 16, 32, 64):
    lp, dp = lora_params(d, k, r), dense_params(d, k)
    print(f"  r={r:<3} one 4096x4096 layer: {lp:>8,} trainable vs {dp:,} full  ({lp/dp*100:.2f}%)")

whole = lora_params(d, k, 16) * 7 * 32   # ~7 all-linear modules x ~32 layers
print(f"  r=16 all-linear across a 7B: ~{whole/1e6:.1f}M trainable (~{whole/7e9*100:.2f}% of 7B) -> a tiny adapter file")

# Output:
#   r=8   one 4096x4096 layer:   65,536 trainable vs 16,777,216 full  (0.39%)
#   r=16  one 4096x4096 layer:  131,072 trainable vs 16,777,216 full  (0.78%)
#   r=32  one 4096x4096 layer:  262,144 trainable vs 16,777,216 full  (1.56%)
#   r=64  one 4096x4096 layer:  524,288 trainable vs 16,777,216 full  (3.12%)
#   r=16 all-linear across a 7B: ~29.4M trainable (~0.42% of 7B) -> a tiny adapter file

**Explanation**

A pure arithmetic harness, no model involved. It compares the two-matrix LoRA cost `r*(d+k)` against the dense cost `d*k` for one square layer, sweeps the rank, then scales the rank-16 figure up to an all-linear 7B to show the adapter is a rounding error on the full model.

**How the code works - step by step**
- **`lora_params(d, k, r)`** - returns `r*(d+k)`, the combined size of the two adapter matrices A (r x k) and B (d x r).
- **`dense_params(d, k)`** - returns `d*k`, what a full weight update would cost.
- **the `for r in (8,16,32,64)` loop** - for a 4096x4096 layer, prints trainable vs full and the percentage, showing it grows linearly with rank.
- **`whole = lora_params(d,k,16) * 7 * 32`** - approximates a rank-16 adapter across ~7 linear modules over ~32 layers of a 7B, then prints it as a percentage of 7B.

**In one line:** trainable params scale as `r*(d+k)` - a fraction of a percent of the dense layer, even summed over the whole model.

**What you'll see:** A four-row table where rank 8/16/32/64 costs 0.39% / 0.78% / 1.56% / 3.12% of one 16.8M-parameter layer, plus a closing line: a rank-16 all-linear adapter across a 7B is ~29.4M params (~0.42% of the model) - a tiny adapter file.

## 2 - Budget the VRAM: fit a 7B on a 16GB GPU

The cold-open of this lesson - a 7B that supposedly needs a data-center GPU running on a free T4 - is really just an arithmetic claim. This cell quantifies it: rough training VRAM per method, and the smallest GPU that holds each. It shows exactly where full fine-tuning falls off the free tier and QLoRA lands back on it.

In [ ]:
# QLoRA VRAM - how a 7B that needs 120GB for full fine-tuning fits a 16GB GPU.
def vram(params_b, method):
    return {"full": params_b*16, "lora": params_b*3.4, "qlora": params_b*1.0 + 3,
            "qlora+unsloth": params_b*0.7 + 2}[method]   # rough training GB

GPUS = [("T4", 16), ("L4", 24), ("A100-40", 40), ("A100-80", 80)]
def fits(v):
    for name, cap in GPUS:
        if v <= cap:
            return name
    return "multi-GPU"

print("  7B model, training VRAM by method + smallest GPU that fits:")
for m in ("full", "lora", "qlora", "qlora+unsloth"):
    v = vram(7, m)
    print(f"    {m:14s} ~{v:6.1f} GB -> {fits(v)}")
print("  QLoRA (+Unsloth) drops a 7B onto a free 16GB T4; full fine-tuning needs multi-GPU.")

# Output:
#   7B model, training VRAM by method + smallest GPU that fits:
#     full           ~ 112.0 GB -> multi-GPU
#     lora           ~  23.8 GB -> L4
#     qlora          ~  10.0 GB -> T4
#     qlora+unsloth  ~   6.9 GB -> T4
#   QLoRA (+Unsloth) drops a 7B onto a free 16GB T4; full fine-tuning needs multi-GPU.

**Explanation**

A back-of-envelope estimator, not a real allocation. It multiplies parameter count (in billions) by a per-method bytes-per-parameter factor to get training GB, then walks a fixed GPU list from smallest to largest to report the first one that fits.

**How the code works - step by step**
- **`vram(params_b, method)`** - a small lookup of per-method costs: full ~16 bytes/param (weights + grads + optimizer states), LoRA ~3.4x, QLoRA ~1.0x + 3GB overhead, +Unsloth ~0.7x + 2GB.
- **`GPUS`** - the capacity ladder: T4 16GB, L4 24GB, A100-40, A100-80.
- **`fits(v)`** - returns the smallest GPU whose capacity holds `v`, else "multi-GPU".
- **the method loop** - for a 7B, prints VRAM and target GPU for full / lora / qlora / qlora+unsloth.

**In one line:** VRAM = params x method-factor -> the first GPU on the ladder that holds it.

**What you'll see:** For a 7B: full ~112GB needs multi-GPU, LoRA ~23.8GB fits an L4, QLoRA ~10GB and QLoRA+Unsloth ~6.9GB both fit a free 16GB T4 - closing with the line that QLoRA (+Unsloth) drops a 7B onto a T4 while full fine-tuning needs multi-GPU.

## 3 - Pick the right LoRA variant flag

Plain LoRA is the baseline; four variants each improve it with a single flag, but only in their lane. This cell encodes the 2026 defaults as a recommender - always DoRA, add rsLoRA at high rank - and flags the two traps: PiSSA fails for reasoning/RL, and LoRA+ conflicts with rsLoRA.

In [ ]:
# LoRA VARIANTS - which single-flag upgrade to turn on, and when.
def recommend_variant(rank, goal):
    flags = {"use_dora": True}          # DoRA is a free upgrade in 2026 - default on
    notes = ["use_dora=True  (+1-3% quality, zero inference overhead)"]
    if rank >= 32:
        flags["use_rslora"] = True
        notes.append("use_rslora=True  (stable gradients - essential at rank >= 32)")
    if goal == "reasoning":
        notes.append("PiSSA helps SFT but FAILS in GRPO/RL - do not use it for reasoning")
    if goal == "speed-experiment":
        notes.append("LoRA+ (higher LR on B) is experimental and conflicts with rsLoRA")
    return flags, notes

for rank, goal in [(16, "style"), (64, "domain"), (16, "reasoning")]:
    flags, notes = recommend_variant(rank, goal)
    print(f"  rank={rank:<3} goal={goal:16s} -> {flags}")
    for note in notes:
        print(f"      - {note}")

# Output:
#   rank=16  goal=style            -> {'use_dora': True}
#       - use_dora=True  (+1-3% quality, zero inference overhead)
#   rank=64  goal=domain           -> {'use_dora': True, 'use_rslora': True}
#       - use_dora=True  (+1-3% quality, zero inference overhead)
#       - use_rslora=True  (stable gradients - essential at rank >= 32)
#   rank=16  goal=reasoning        -> {'use_dora': True}
#       - use_dora=True  (+1-3% quality, zero inference overhead)
#       - PiSSA helps SFT but FAILS in GRPO/RL - do not use it for reasoning

**Explanation**

A rules-as-code decision function, not a training call. Given a rank and a goal it returns the flags to set plus human-readable notes, baking the research consensus into branch conditions so you read the guidance off the output.

**How the code works - step by step**
- **`recommend_variant(rank, goal)`** - starts with `use_dora=True` always (the free +1-3% quality upgrade with zero inference cost).
- **the `if rank >= 32` branch** - adds `use_rslora=True` for gradient stability at high rank.
- **the `goal == "reasoning"` branch** - warns that PiSSA helps SFT but fails in GRPO/RL.
- **the `goal == "speed-experiment"` branch** - warns LoRA+ is experimental and conflicts with rsLoRA.
- **the scenario loop** - runs three (rank, goal) pairs and prints the resulting flags plus notes.

**In one line:** DoRA always, rsLoRA at rank>=32, and stay out of PiSSA-for-reasoning and LoRA+-with-rsLoRA.

**What you'll see:** Three scenarios: rank-16 style and rank-16 reasoning both return just `{'use_dora': True}` (the reasoning case adds a PiSSA warning), while rank-64 domain returns `{'use_dora': True, 'use_rslora': True}` with the high-rank stability note.

## 4 - Match the alignment method to your goal and data

Alignment is not one trainer but an ordered three: SFT for the behavioral foundation, DPO for subjective preferences, GRPO for verifiable reasoning. This cell maps a (goal, data) pair to the right method and reminds you what each one needs - preference pairs for DPO, a verifier for GRPO.

In [ ]:
# ALIGNMENT - match the method to the goal + the data (SFT -> DPO -> GRPO).
def pick_alignment(goal, data):
    table = {
        ("format",    "pairs"):       ("SFT",  "input-output pairs teach the behavioral foundation - always first"),
        ("style",     "preferences"): ("DPO",  "chosen/rejected pairs; no reward model (TRL disables the adapter for ref logits)"),
        ("reasoning", "verifiable"):  ("GRPO", "sample G completions, reward correct answers, normalize in-group; powers DeepSeek-R1"),
        ("memory",    "preferences"): ("ORPO", "single-stage SFT+alignment, no reference model (trl.experimental in v1.0)"),
    }
    return table.get((goal, data), ("SFT", "default: start with supervised fine-tuning"))

for goal, data in [("format","pairs"), ("style","preferences"), ("reasoning","verifiable"), ("memory","preferences")]:
    method, why = pick_alignment(goal, data)
    print(f"  goal={goal:10s} data={data:12s} -> {method:5s} | {why}")
print("  Standard pipeline: SFT (foundation) -> DPO (subjective quality) -> GRPO (verifiable reasoning).")

# Output:
#   goal=format     data=pairs        -> SFT   | input-output pairs teach the behavioral foundation - always first
#   goal=style      data=preferences  -> DPO   | chosen/rejected pairs; no reward model (TRL disables the adapter for ref logits)
#   goal=reasoning  data=verifiable   -> GRPO  | sample G completions, reward correct answers, normalize in-group; powers DeepSeek-R1
#   goal=memory     data=preferences  -> ORPO  | single-stage SFT+alignment, no reference model (trl.experimental in v1.0)
#   Standard pipeline: SFT (foundation) -> DPO (subjective quality) -> GRPO (verifiable reasoning).

**Explanation**

A lookup-table selector, not a trainer. It keys on the combination of what you want and the data you have, returning the method name plus the one fact that makes it work (or the TRL v1.0 status), so the whole SFT->DPO->GRPO ladder reads out of one dict.

**How the code works - step by step**
- **`pick_alignment(goal, data)`** - a dict keyed on `(goal, data)` pairs: format+pairs->SFT, style+preferences->DPO, reasoning+verifiable->GRPO, memory+preferences->ORPO, with an SFT default fallback.
- Each entry carries the key detail: DPO disables the LoRA adapter to get reference logits (no second model), GRPO samples G completions and normalizes rewards in-group, ORPO is now `trl.experimental`.
- **the scenario loop** - runs the four pairs and prints method + rationale, then a closing line stating the standard order.

**In one line:** goal+data -> the method, following SFT (foundation) -> DPO (preferences) -> GRPO (verifiable reasoning).

**What you'll see:** Four rows mapping (format, pairs)->SFT, (style, preferences)->DPO, (reasoning, verifiable)->GRPO, (memory, preferences)->ORPO, each with a one-line why, closed by the standard-pipeline summary line.

## 5 - Choose a framework, then a serving pattern

Four training frameworks cover the field by hardware and skill level, and serving splits on task count: merge one adapter into the base, or keep many adapters separate and route per request. This cell encodes both decisions - the trainer picker and the serving picker - so the multi-LoRA production pattern is explicit.

In [ ]:
# FRAMEWORK + SERVING - pick the trainer, then the serving pattern.
def pick_framework(multi_gpu, want_zero_code, want_max_control):
    if multi_gpu:        return ("Axolotl", "production multi-GPU via YAML + FSDP2/DeepSpeed")
    if want_zero_code:   return ("LLaMA-Factory", "zero-code web UI, 100+ models (Unsloth backend)")
    if want_max_control: return ("torchtune", "pure PyTorch, maximum control")
    return ("Unsloth + TRL", "single-GPU speed: ~2x faster, ~70% less VRAM")

def pick_serving(n_tasks):
    if n_tasks == 1:
        return "merge_and_unload -> one merged model (zero adapter overhead)"
    return f"vLLM multi-LoRA: ONE base + {n_tasks} adapters (--enable-lora --lora-modules), swapped per request"

for label, args in [("Single GPU, code", (False, False, False)),
                    ("4x A100 cluster", (True, False, False)),
                    ("Analyst, no code", (False, True, False))]:
    fw, why = pick_framework(*args)
    print(f"  {label:20s} -> {fw:15s} | {why}")
print(f"  Serving 1 task : {pick_serving(1)}")
print(f"  Serving 3 tasks: {pick_serving(3)}")

# Output:
#   Single GPU, code     -> Unsloth + TRL   | single-GPU speed: ~2x faster, ~70% less VRAM
#   4x A100 cluster      -> Axolotl         | production multi-GPU via YAML + FSDP2/DeepSpeed
#   Analyst, no code     -> LLaMA-Factory   | zero-code web UI, 100+ models (Unsloth backend)
#   Serving 1 task : merge_and_unload -> one merged model (zero adapter overhead)
#   Serving 3 tasks: vLLM multi-LoRA: ONE base + 3 adapters (--enable-lora --lora-modules), swapped per request

**Explanation**

Two small routing functions, no serving actually stood up. `pick_framework` branches on hardware and preference flags; `pick_serving` branches on how many task adapters you have, returning merge for one and vLLM multi-LoRA for many.

**How the code works - step by step**
- **`pick_framework(multi_gpu, want_zero_code, want_max_control)`** - priority branches: multi-GPU->Axolotl, zero-code->LLaMA-Factory, max-control->torchtune, else Unsloth+TRL.
- **`pick_serving(n_tasks)`** - one task returns `merge_and_unload` (zero adapter overhead); many returns vLLM multi-LoRA (one base + N adapters swapped per request).
- **the framework loop** - runs three hardware profiles and prints the chosen trainer.
- **the two `pick_serving` calls** - contrast serving 1 task vs 3 tasks.

**In one line:** framework follows the hardware; serving follows the task count - merge for one, multi-LoRA for many.

**What you'll see:** Single-GPU->Unsloth+TRL, 4x A100->Axolotl, no-code analyst->LLaMA-Factory; then serving 1 task = merge into one model, serving 3 tasks = one vLLM base plus 3 adapters routed per request.

## 6 - Cost out the India open-weight path

Open weights make India's fine-tuning story both cheap and sovereign: Sarvam's Indic tokenizer roughly halves the tokens on Hindi, and IndiaAI/Jarvislabs GPUs cost a fraction of hyperscaler rates. This cell turns a rupee budget into a concrete stack - model, GPU, and deploy path - scaling from free up to subsidized production compute.

In [ ]:
# INDIA - budget decides the stack; Sarvam + Unsloth + IndiaAI is the cheap path.
def india_plan(budget_inr):
    if budget_inr < 100:
        return ("free Colab T4 + Unsloth", "Sarvam-1 (2B) or Gemma 3 4B, QLoRA", "export GGUF -> Ollama (local, free)")
    if budget_inr <= 600:
        hrs = round(budget_inr / 108, 1)   # Jarvislabs A100 ~ Rs 108/hr
        return (f"Jarvislabs A100 (~Rs 108/hr, ~{hrs} hrs)", "Sarvam-1 / Sarvam-M (24B) QLoRA", "IndicAlign data (free) + GGUF export")
    return ("IndiaAI Mission GPUs (~Rs 65-92/hr, subsidized)", "Sarvam-M 24B QLoRA", "production serving on Indian cloud")

for b in (0, 500, 5000):
    stack, model, deploy = india_plan(b)
    print(f"  budget Rs {b:<5} -> {stack}")
    print(f"      model: {model}   deploy: {deploy}")
print("  Sarvam-1's Indic tokenizer costs ~half the tokens of Llama on Hindi -> cheaper training + inference.")

# Output:
#   budget Rs 0     -> free Colab T4 + Unsloth
#       model: Sarvam-1 (2B) or Gemma 3 4B, QLoRA   deploy: export GGUF -> Ollama (local, free)
#   budget Rs 500   -> Jarvislabs A100 (~Rs 108/hr, ~4.6 hrs)
#       model: Sarvam-1 / Sarvam-M (24B) QLoRA   deploy: IndicAlign data (free) + GGUF export
#   budget Rs 5000  -> IndiaAI Mission GPUs (~Rs 65-92/hr, subsidized)
#       model: Sarvam-M 24B QLoRA   deploy: production serving on Indian cloud
#   Sarvam-1's Indic tokenizer costs ~half the tokens of Llama on Hindi -> cheaper training + inference.

**Explanation**

A budget-to-stack planner, pure Python with no calls out. It thresholds the rupee amount into three tiers and, in the middle tier, divides the budget by the Jarvislabs hourly rate to show how many A100 hours it buys.

**How the code works - step by step**
- **`india_plan(budget_inr)`** - three tiers: under 100 -> free Colab T4 + Unsloth; up to 600 -> Jarvislabs A100 with an `hrs = budget/108` hours estimate; above -> subsidized IndiaAI Mission GPUs.
- Each tier names the Sarvam model that fits (Sarvam-1 2B, Sarvam-M 24B) and the deploy path (GGUF -> Ollama for local, Indian cloud for production).
- **the budget loop** - runs 0, 500, 5000 and prints the stack, model, and deploy for each.
- The closing line notes the Indic tokenizer roughly halves Hindi token cost.

**In one line:** budget thresholds -> a named Sarvam-on-Indian-GPU stack, from a free T4 up to subsidized IndiaAI.

**What you'll see:** Three budget rows: Rs 0 = free Colab T4 + Unsloth (Sarvam-1, GGUF->Ollama), Rs 500 = Jarvislabs A100 for ~4.6 hrs, Rs 5000 = subsidized IndiaAI GPUs - closing with the Indic-tokenizer cost note.

Across eight cells you moved from "fine-tuning needs a data-center GPU" to a costed, decision-driven pipeline you can run on a free T4: LoRA trains a fraction of a percent of the weights (Cell 3), QLoRA + Unsloth collapse a 7B onto 16GB of VRAM (Cell 4), and single-flag variants, the SFT→DPO→GRPO ladder, framework choice, and the India stack are all encoded as small lookup functions (Cells 5-8). Each decision here is the one you would carry into a real TRL v1.0 or Unsloth training run. Next, Lesson 9.5 derives the LoRA/QLoRA math this lesson used at face value, and 9.6 takes the resulting adapter into evaluation, merging, and deployment.